## **PySpark**

## CUSTOMERS

**Ex 1:Create a DataFrame for customers**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
          .option("inferSchema", True)\
          .option("header", True)\
          .load("/Volumes/databricksgovi/bronze/bronze_volume/customers/")

- First we need to specify the format of the file
- Second we need to inferSchema as True
- Third we need to add header as True
- Finally we need to pass the path of the folder where the csv file exists

**Ex 2: Display the DataFrame**

In [0]:
display(df)

****

**Ex 3: Print the Schema**

In [0]:
df.printSchema()

### **PERFORMING BASIC TRANSFORMATIONS**

**Ex 1:Convert all the names into uppercase present in name column**

- To perform this transformation we need to import all the functions and types from sql

_withColumn() function is a DataFrame transformation used to add a new column or modify an existing one within a DataFrame._

In [0]:
df = df.withColumn("name", upper(col("name")))
display(df)

**Ex 2: Create a new domain column based on customer email_id's**

In [0]:
df = df.withColumn("domain", split(col("email"),"@")[1])
display(df)

**Ex 3: Display the avg count of the customers based on domain**

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("total_customer")))

**Ex 4: Now the Apply the sorting and visualization**

In [0]:
display(
  df.groupBy("domain")
    .agg(
      count(col("customer_id")).alias("total_customer")
    )
    .sort(
      col("total_customer").desc()
    )
)

Databricks visualization. Run in Databricks to view.

**Ex 5: Create a new process date column with current time stamp**

In [0]:
df = df.withColumn("processDate", current_timestamp())
display(df)

### **What is UPSERT**
_In Databricks with PySpark, UPSERT (Update/Insert) is an efficient data management technique, primarily achieved using the MERGE INTO command on Delta Lake tables, to synchronize source data (DataFrame/Table) with a target Delta table by inserting new rows and updating existing ones in a single operation, simplifying Change Data Capture (CDC) and ensuring data consistency without full table rewrites_

- .whenMatchedUpdateAll()
- .whenNotMatchedInsertAll()

### **Push the data into the Silver using UPSERT**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricksgovi.silver.customer_enr"):

    dlt_obj = DeltaTable.forName(spark,"databricksgovi.silver.customer_enr")
    dlt_obj.alias("trg").merge(df.alias("src"),"trg.customer_id == src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksgovi.silver.customer_enr")

**Check data upserted or not**

In [0]:
%sql
SELECT * FROM databricksgovi.silver.customer_enr;

### **PRODUCTS**

**Ex 1:Create Dataframe for products**

In [0]:
df_prod = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/databricksgovi/bronze/bronze_volume/products/")

In [0]:
display(df_prod)

**Ex 2: Add Process date column into Products table**

In [0]:
df_prod = df_prod.withColumn("processDate", current_timestamp())
display(df_prod)

**Ex 3: Apply aggregation and display Avg of price on each category**

In [0]:
display(df_prod.groupBy("category").agg(avg("price").alias("avg_price")))

Databricks visualization. Run in Databricks to view.

**Ex 4: Push the products data into silver layer by applying UPSERT**

In [0]:
if spark.catalog.tableExists("databricksgovi.silver.products_enr"):

    dlt_obj = DeltaTable.forName(spark,"databricksgovi.silver.products_enr")
    dlt_obj.alias("trg").merge(df_prod.alias("src"),"trg.product_id == src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_prod.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksgovi.silver.products_enr")

In [0]:
%sql
SELECT * from databricksgovi.silver.products_enr;

### **STORES**

**Ex 1: Create a dataframe for stores**

In [0]:
df_store = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/databricksgovi/bronze/bronze_volume/stores/")

In [0]:
display(df_store)

**Ex 2: Create a process date column in stores**

In [0]:
df_store = df_store.withColumn("processDate", current_timestamp())

In [0]:
display(df_store)

**Ex 3:Remove the underscore from store_name column**

In [0]:
df_store = df_store.withColumn("store_name", regexp_replace(col("store_name"),"_",""))
display(df_store)

**Ex 4: Push all the store data into Silver layer using UPSERT**

In [0]:
if spark.catalog.tableExists("databricksgovi.silver.stores_enr"):

    dlt_obj = DeltaTable.forName(spark,"databricksgovi.silver.stores_enr")
    dlt_obj.alias("trg").merge(df_store.alias("src"),"trg.store_id == src.store_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_store.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksgovi.silver.stores_enr")

In [0]:
%sql
SELECT * from databricksgovi.silver.stores_enr;

### **SALES**

**Ex 1: Create dataframe for sales**

In [0]:
df_sales = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/databricksgovi/bronze/bronze_volume/sales/")
display(df_sales)

**Ex 2: Find and create a column for process date and price per sale using quantity and sales**

In [0]:
df_sales = df_sales.withColumn("pricePerSale",round(col("total_amount")/col("quantity"),2))
df_sales = df_sales.withColumn("processDate", current_timestamp())
display(df_sales)

**Ex 3: Push the sales data into silver layer using UPSERT**

In [0]:
if spark.catalog.tableExists("databricksgovi.silver.sales_enr"):

    dlt_obj = DeltaTable.forName(spark,"databricksgovi.silver.sales_enr")
    dlt_obj.alias("trg").merge(df_sales.alias("src"),"trg.sales_id == src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_sales.write.format("delta")\
        .mode("append")\
        .saveAsTable("databricksgovi.silver.sales_enr")

In [0]:
%sql
SELECT * FROM databricksgovi.silver.sales_enr;

## **SPARK SQL**
_Spark SQL is a module within Apache Spark designed for processing structured data. It offers a powerful and flexible way to interact with data, providing both a SQL interface and a DataFrame API._

In [0]:
df_pro_sil = spark.sql("select * from databricksgovi.silver.products_enr")

In [0]:
display(df_pro_sil)

**Ex 1: Create Temprary views for products**

In [0]:
df_pro_sil.createOrReplaceTempView("temp_products")

**Ex 2: Apply condition statement using Spark sql for category column**

In [0]:
df_pro_sil = spark.sql("""
    SELECT *,
        CASE 
            WHEN category = 'Toys' THEN 'YES' ELSE 'NO' END AS flag
    FROM temp_products
""")

In [0]:
display(df_pro_sil)

### **Pyspark UDF/Custom Fuctions**

In [0]:
def convey(p_input):
    return "Hey "+str(p_input)

In [0]:
udf_convey = udf(convey)

**Ex 1: Create a convey column and append the provided value**

In [0]:
df_pro_sil = df_pro_sil.withColumn("convey", udf_convey(col("flag")))
display(df_pro_sil)